# DEG Analysis — CellTypist Low (fine-grained) Cell Types
Differential expression across all 14 cell types annotated by `Immune_All_Low.pkl`.

## 1. Imports & settings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

## 2. Load data

In [ ]:
adata = sc.read_h5ad('bone_marrow_celltypist.h5ad')
adata

In [ ]:
# Confirm celltypist_low labels are present and inspect composition
print(adata.obs['celltypist_low'].value_counts())

## 3. `method` and `corr_method` — parameter guide

### `method`
Selects the **statistical test** used to score each gene per group.

| Value | What it does | When to use |
|---|---|---|
| `'wilcoxon'` *(recommended)* | Wilcoxon rank-sum test (Mann-Whitney U). Non-parametric: compares rank distributions, not means. | Default choice for scRNA-seq — robust to zero-inflation and non-normality |
| `'t-test'` | Welch's t-test on log-normalised expression. Fast, parametric. | Quick first pass; assumes approximately normal residuals |
| `'t-test_overestim_var'` | t-test with inflated variance estimate. More conservative. | When you want fewer false positives at the cost of sensitivity |
| `'logreg'` | One-vs-rest logistic regression (L1-penalised). Returns classifier coefficients, not p-values. | Identifying the smallest discriminative gene set; no p-value / FDR output |

**Rule of thumb**: use `'wilcoxon'` unless you have a specific reason to switch.

---

### `corr_method`
Selects the **multiple-testing correction** applied to raw p-values (not used with `'logreg'`).

| Value | What it does | When to use |
|---|---|---|
| `'benjamini-hochberg'` *(default)* | Controls the **false discovery rate (FDR)**. Ranks p-values and applies a stepwise threshold — less conservative, more power. | Standard for exploratory scRNA-seq DEG analysis |
| `'bonferroni'` | Multiplies each p-value by the total number of tests. Very conservative — controls **family-wise error rate (FWER)**. | When even a single false positive is unacceptable (e.g. validation experiments) |

With ~20 000 genes tested per cell type, Bonferroni will reject almost everything; BH is the practical default.

## 4. Run DEG analysis

Each cell type in `celltypist_low` is tested against all other cells (one-vs-rest).

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby='celltypist_low',
    method='wilcoxon',      # Wilcoxon rank-sum — robust for scRNA-seq
    corr_method='benjamini-hochberg',  # FDR correction
    tie_correct=True,       # correct for ties in the rank-sum test
    pts=True,               # store fraction of cells expressing each gene per group
)
print("DEG analysis complete.")

## 5. Visualise — dotplot of top 5 markers per cell type

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby='celltypist_low',
    standard_scale='var',   # scale each gene 0-1 across groups for visual contrast
    n_genes=5,
    title='Top 5 DEGs per celltypist_low cell type',
    figsize=(28, 8),
)

## 6. Extract results as a tidy dataframe

In [ ]:
cell_types = adata.obs['celltypist_low'].cat.categories.tolist()

deg_results = []
for ct in cell_types:
    df = sc.get.rank_genes_groups_df(
        adata,
        group=ct,
        pval_cutoff=0.05,
        log2fc_min=0.5,
    )
    df.insert(0, 'cell_type', ct)
    deg_results.append(df)

deg_df = pd.concat(deg_results, ignore_index=True)
print(f"Total significant DEGs (FDR < 0.05, log2FC ≥ 0.5): {len(deg_df)}")
deg_df.head(10)

In [ ]:
# DEG count per cell type
deg_df.groupby('cell_type').size().sort_values(ascending=False).rename('n_DEGs')

## 7. Top 5 DEGs per cell type (names only)

In [ ]:
for ct in cell_types:
    top5 = sc.get.rank_genes_groups_df(adata, group=ct).head(5)['names'].tolist()
    print(f"{ct}: {', '.join(top5)}")

## 8. UMAP coloured by top marker of each cell type

In [ ]:
top_markers = []
for ct in cell_types:
    df = sc.get.rank_genes_groups_df(adata, group=ct)
    # pick first gene present in adata.var_names
    for g in df['names']:
        if g in adata.var_names:
            top_markers.append(g)
            break

sc.pl.umap(
    adata,
    color=top_markers,
    ncols=4,
    color_map='RdBu_r',
    vmin=-0.5,
)

## 9. Save DEG table

In [ ]:
deg_df.to_csv('deg_celltypist_low.csv', index=False)
print("Saved: deg_celltypist_low.csv")

> **Note on p-value interpretation**: individual cells are treated as independent observations, which inflates statistical power and makes p-values appear artificially low. For publication-grade comparisons across conditions/samples, use pseudo-bulk DEG methods (e.g. `pydeseq2`) that aggregate counts per donor before testing.